# 00 — Setup Verification

Notebook ini memverifikasi bahwa seluruh service Fase 2 sudah berjalan dan dapat diakses:

| Service    | URL                           | Cek |
|------------|-------------------------------|-----|
| ClickHouse | http://clickhouse:8123/ping   | ✅/❌ |
| MLflow     | http://mlflow:5000            | ✅/❌ |
| Jupyter    | (notebook ini sendiri)        | ✅   |

**Jalankan semua cell dari atas ke bawah** — jika tidak ada error, environment siap digunakan.

## 1. Cek Versi Library

In [1]:
import sys, importlib

libs = [
    'clickhouse_connect', 'pandas', 'numpy', 'sklearn',
    'mlxtend', 'mlflow', 'matplotlib', 'seaborn', 'plotly'
]

print(f"Python  : {sys.version.split()[0]}")
for lib in libs:
    try:
        mod = importlib.import_module(lib)
        ver = getattr(mod, '__version__', 'n/a')
        print(f"  {lib:<22}: {ver}")
    except ImportError as e:
        print(f"  {lib:<22}: TIDAK DITEMUKAN — {e}")

Python  : 3.11.6
  clickhouse_connect    : <module 'clickhouse_connect.__version__' from '/opt/conda/lib/python3.11/site-packages/clickhouse_connect/__version__.py'>
  pandas                : 2.1.1
  numpy                 : 1.26.4
  sklearn               : 1.8.0
  mlxtend               : 0.23.4
  mlflow                : 3.10.1
  matplotlib            : 3.8.0
  seaborn               : 0.13.0
  plotly                : 6.6.0


## 2. Koneksi ke ClickHouse

In [2]:
import sys
sys.path.insert(0, '/home/jovyan/work/notebooks')

from utils import get_ch_client, read_sql, show_ch_tables

client = get_ch_client()
result = client.query('SELECT version() AS ch_version').result_rows
print(f"✅ ClickHouse tersambung — versi: {result[0][0]}")

✅ ClickHouse tersambung — versi: 24.3.18.7


In [3]:
# Tampilkan tabel yang tersedia di tiap layer
for schema in ['bronze', 'silver', 'gold']:
    df = show_ch_tables(schema, client)
    print(f"\n── {schema.upper()} ({len(df)} tabel) ──")
    print(df.to_string(index=False))


── BRONZE (8 tabel) ──
            name    engine total_rows total_bytes
bronze_customers MergeTree      10.00    1.38 KiB
  bronze_reviews MergeTree      30.00    2.98 KiB
    bronze_sales MergeTree      10.00    1.69 KiB
  bronze_targets MergeTree      25.00    1.08 KiB
       customers MergeTree      10.00    1.38 KiB
         reviews MergeTree      30.00    2.98 KiB
           sales MergeTree      10.00    1.69 KiB
         targets MergeTree      25.00    1.08 KiB

── SILVER (4 tabel) ──
            name    engine total_rows total_bytes
silver_customers MergeTree      10.00    1.45 KiB
  silver_reviews MergeTree      30.00    3.29 KiB
    silver_sales MergeTree      10.00    2.00 KiB
  silver_targets MergeTree      25.00    1.27 KiB

── GOLD (3 tabel) ──
               name    engine total_rows total_bytes
    gold_branch_kpi MergeTree       5.00    1.62 KiB
gold_review_summary MergeTree       5.00    1.52 KiB
   gold_sales_daily MergeTree      10.00    2.01 KiB


In [4]:
# Sample data dari gold layer
df_gold = read_sql('SELECT * FROM gold.gold_sales_daily LIMIT 5', client)
print(f"gold_sales_daily — {len(df_gold)} baris (sample)")
df_gold

gold_sales_daily — 5 baris (sample)


,order_date,order_year,order_month,branch,total_orders,total_revenue,avg_order_value,total_items_sold,orders_elektronik,orders_aksesoris,orders_komponen,rev_elektronik,rev_aksesoris,rev_komponen
0,2024-01-05,2024,1,Pusat,1,7500000.00,7500000.0,1,1,0,0,7500000.00,0.00,0.00
1,2024-01-07,2024,1,Bandung,1,700000.00,700000.0,2,0,1,0,0.00,700000.00,0.00
2,2024-01-10,2024,1,Pusat,1,2800000.00,2800000.0,1,1,0,0,2800000.00,0.00,0.00
3,2024-01-12,2024,1,Surabaya,1,850000.00,850000.0,1,0,1,0,0.00,850000.00,0.00
4,2024-01-15,2024,1,Selatan,1,1200000.00,1200000.0,1,0,1,0,0.00,1200000.00,0.00


## 3. Koneksi ke MLflow

In [5]:
import mlflow, os

tracking_uri = os.getenv('MLFLOW_TRACKING_URI', 'http://mlflow:5000')
mlflow.set_tracking_uri(tracking_uri)

client_mf = mlflow.tracking.MlflowClient()
experiments = client_mf.search_experiments()

print(f"✅ MLflow tersambung — URI: {tracking_uri}")
print(f"   Jumlah experiment: {len(experiments)}")

if experiments:
    for exp in experiments:
        print(f"   - [{exp.experiment_id}] {exp.name}")

✅ MLflow tersambung — URI: http://mlflow:5000
   Jumlah experiment: 1
   - [0] Default


In [6]:
# Test logging sederhana ke MLflow
with mlflow.start_run(run_name='setup_verification') as run:
    mlflow.set_tag('notebook', '00_setup_verification')
    mlflow.log_param('test_param', 'ok')
    mlflow.log_metric('test_metric', 1.0)
    run_id = run.info.run_id

print(f"✅ MLflow logging berhasil — run_id: {run_id}")
print(f"   Lihat di: {tracking_uri}")

🏃 View run setup_verification at: http://mlflow:5000/#/experiments/0/runs/408f0f46f75c408782adf894e35c1206
🧪 View experiment at: http://mlflow:5000/#/experiments/0
✅ MLflow logging berhasil — run_id: 408f0f46f75c408782adf894e35c1206
   Lihat di: http://mlflow:5000


## 4. Ringkasan

Jika semua cell di atas berjalan tanpa error, environment Fase 2 siap.

**Notebook selanjutnya:**

| Notebook | Topik | Model |
|----------|-------|-------|
| `01_clustering_customer_rfm.ipynb`     | Segmentasi pelanggan (RFM)   | K-Means |
| `02_classification_order_status.ipynb` | Prediksi status order        | Random Forest |
| `03_regression_revenue_forecast.ipynb` | Forecast pendapatan          | Linear/Ridge Regression |
| `04_association_market_basket.ipynb`   | Analisis keranjang belanja   | FP-Growth |